In [1]:
import os
import zipfile
import subprocess

def download_kaggle_dataset(dataset_name: str, download_path: str):
    os.makedirs(download_path, exist_ok=True)

    result = subprocess.run(
        [
            "kaggle",
            "datasets",
            "download",
            "-d",
            dataset_name,
            "-p",
            download_path,
        ],
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        raise Exception(result.stderr)

    # Extract zip files
    for file in os.listdir(download_path):
        if file.endswith(".zip"):
            zip_path = os.path.join(download_path, file)

            with zipfile.ZipFile(zip_path, "r") as zip_ref:
                zip_ref.extractall(download_path)

    print(f"Dataset downloaded to: {os.path.abspath(download_path)}")

In [2]:
dataset_name = 'uciml/sms-spam-collection-dataset'
download_path = './datasets'
download_kaggle_dataset(dataset_name, download_path)

Dataset downloaded to: /content/datasets


In [3]:
import pandas as pd

df = pd.read_csv(
    "./datasets/spam.csv",
    encoding="latin-1"
)

# Keep only useful columns
df = df[["v1", "v2"]]

# Rename columns
df.columns = ["label", "message"]

df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
# Data Cleaning and Preprocessing
import re
import nltk
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [5]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

wl = WordNetLemmatizer()

In [6]:
corpus = []
for i in range(len(df)):
    review = re.sub("[^a-zA-Z]", " ", df["message"][i])
    review = review.lower()
    review = review.split()

    review = [
        wl.lemmatize(word)
        for word in review
        if word not in set(stopwords.words("english"))
    ]
    review = " ".join(review)
    corpus.append(review)

In [7]:
# Create Bag Of Words model
from sklearn.feature_extraction.text import CountVectorizer
# For Binary BOW enable binary=True
cv = CountVectorizer(max_features=2500, ngram_range=(1, 2))

In [8]:
# Independent Features
X = cv.fit_transform(corpus).toarray()

In [9]:
import numpy as np

np.set_printoptions(
    edgeitems=30,
    linewidth=100000,
    formatter={"float_kind": lambda x: "%.3g" % x}
)

In [10]:
cv.vocabulary_

{'go': np.int64(811),
 'point': np.int64(1614),
 'crazy': np.int64(447),
 'available': np.int64(112),
 'bugis': np.int64(227),
 'great': np.int64(856),
 'world': np.int64(2436),
 'la': np.int64(1096),
 'cine': np.int64(345),
 'got': np.int64(847),
 'wat': np.int64(2336),
 'ok': np.int64(1477),
 'lar': np.int64(1108),
 'joking': np.int64(1053),
 'wif': np.int64(2399),
 'oni': np.int64(1493),
 'free': np.int64(718),
 'entry': np.int64(616),
 'wkly': np.int64(2422),
 'comp': np.int64(401),
 'win': np.int64(2404),
 'cup': np.int64(455),
 'final': np.int64(683),
 'st': np.int64(1961),
 'may': np.int64(1267),
 'text': np.int64(2090),
 'receive': np.int64(1696),
 'question': np.int64(1662),
 'std': np.int64(1979),
 'txt': np.int64(2200),
 'rate': np.int64(1675),
 'apply': np.int64(77),
 'free entry': np.int64(723),
 'entry wkly': np.int64(618),
 'std txt': np.int64(1980),
 'txt rate': np.int64(2204),
 'rate apply': np.int64(1676),
 'dun': np.int64(571),
 'say': np.int64(1796),
 'early': np.in

In [11]:
# Dependent Feature / Output Feature
y = pd.get_dummies(df["label"])

In [12]:
y

,ham,spam
0,True,False
1,True,False
2,False,True
3,True,False
4,True,False
...,...,...
5567,False,True
5568,True,False
5569,True,False
5570,True,False


In [13]:
y = y.iloc[:, 0].values

In [14]:
y.shape

(5572,)

In [15]:
y

array([ True,  True, False,  True,  True, False,  True,  True, False, False,  True, False, False,  True,  True, False,  True,  True,  True, False,  True,  True,  True,  True,  True,  True,  True,  True,  True,  True, ...,  True,  True,  True,  True,  True, False,  True,  True,  True,  True,  True,  True,  True,  True,  True,  True,  True,  True,  True,  True,  True,  True,  True,  True, False, False,  True,  True,  True,  True])

In [16]:
# Train Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=0
)

In [17]:
from sklearn.naive_bayes import MultinomialNB

In [18]:
spam_detect_model = MultinomialNB().fit(X_train, y_train)

In [19]:
y_pred = spam_detect_model.predict(X_test)

In [20]:
from sklearn.metrics import accuracy_score, classification_report

In [21]:
accuracy_score(y_test, y_pred)

0.9838565022421525

In [22]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.97      0.92      0.94       166
        True       0.99      1.00      0.99       949

    accuracy                           0.98      1115
   macro avg       0.98      0.96      0.97      1115
weighted avg       0.98      0.98      0.98      1115



## Creating The TF-IDF Model

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [24]:
tv = TfidfVectorizer(
    max_features=2500,
    ngram_range=(1, 2)
)

In [25]:
X = tv.fit_transform(corpus).toarray()

In [26]:
tv.vocabulary_

{'go': np.int64(811),
 'point': np.int64(1614),
 'crazy': np.int64(447),
 'available': np.int64(112),
 'bugis': np.int64(227),
 'great': np.int64(856),
 'world': np.int64(2436),
 'la': np.int64(1096),
 'cine': np.int64(345),
 'got': np.int64(847),
 'wat': np.int64(2336),
 'ok': np.int64(1477),
 'lar': np.int64(1108),
 'joking': np.int64(1053),
 'wif': np.int64(2399),
 'oni': np.int64(1493),
 'free': np.int64(718),
 'entry': np.int64(616),
 'wkly': np.int64(2422),
 'comp': np.int64(401),
 'win': np.int64(2404),
 'cup': np.int64(455),
 'final': np.int64(683),
 'st': np.int64(1961),
 'may': np.int64(1267),
 'text': np.int64(2090),
 'receive': np.int64(1696),
 'question': np.int64(1662),
 'std': np.int64(1979),
 'txt': np.int64(2200),
 'rate': np.int64(1675),
 'apply': np.int64(77),
 'free entry': np.int64(723),
 'entry wkly': np.int64(618),
 'std txt': np.int64(1980),
 'txt rate': np.int64(2204),
 'rate apply': np.int64(1676),
 'dun': np.int64(571),
 'say': np.int64(1796),
 'early': np.in

In [27]:
# Train Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=0
)

In [28]:
from sklearn.naive_bayes import MultinomialNB
spam_tfidf_model = MultinomialNB().fit(X_train, y_train)

In [29]:
# Prediction
y_pred = spam_tfidf_model.predict(X_test)

In [30]:
score = accuracy_score(y_test, y_pred)
score

0.968609865470852

In [31]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

       False       0.99      0.80      0.88       166
        True       0.97      1.00      0.98       949

    accuracy                           0.97      1115
   macro avg       0.98      0.90      0.93      1115
weighted avg       0.97      0.97      0.97      1115

